In [1]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
STREAM_HOST = os.environ.get("SOCKET_HOST", "localhost")
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("spark-structured-streaming-preparation").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 17:00:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.2


# Spark Structured Streaming

Structured Streaming treats a **live data stream as an unbounded table** that keeps growing.  
You write the same DataFrame/SQL queries as for batch data -- Spark handles the incremental execution.

| Concept | Description |
|---------|-------------|
| **Source** | Where data arrives: file directory, socket, Kafka, … |
| **Transformation** | Same as batch: `select`, `filter`, `groupBy`, … |
| **Sink** | Where results go: console, file, memory, Kafka |
| **Output mode** | `append` (new rows only), `update` (changed rows), `complete` (full result) |
| **Trigger** | When to run a micro-batch: every N seconds, once (`availableNow`), or continuously |
| **Checkpoint** | State saved to disk for fault tolerance and restart |

> **Notebook note**: File-streaming examples use `trigger(availableNow=True)` to process all  
> existing files and stop -- this makes them runnable without blocking indefinitely.  
> Socket-streaming examples are shown as code patterns; they need `nc -lk 11111` to run.

References:
- https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html

# Conceptual Mapping: DStream API → Structured Streaming

| DStream concept | Structured Streaming equivalent |
|-----------------|----------------------------------|
| `StreamingContext(sc, interval)` | `SparkSession.readStream` |
| `socketTextStream(host, port)` | `.format("socket").option(...)` |
| `textFileStream(dir)` | `.format("text").load(dir)` |
| `DStream.map / filter` | DataFrame `select / filter` |
| `reduceByKey` | `groupBy.agg` (stateless) |
| `reduceByKeyAndWindow` | `withWatermark + window + groupBy` |
| `DStream.foreachRDD` | `writeStream.foreach / foreachBatch` |
| `saveAsTextFiles` | `.writeStream.format("text")` |
| `ssc.start()` | `query.start()` |
| `ssc.awaitTermination()` | `query.awaitTermination()` |

**Key improvements in Structured Streaming:**
- Declarative DataFrame/SQL API instead of RDD lambda chains.
- End-to-end exactly-once guarantees with checkpointing.
- Native event-time windowing and watermarks.
- Continuous mode (near-real-time, sub-millisecond latency) option.

# Example 1: Streaming from a File Directory

Spark watches a directory for new files and processes each one as it arrives.  
The **schema must be specified explicitly** -- Spark cannot infer it from an empty directory.

We use web server log files in `data/weblog_src/`. Each CSV has one column: `request`.

In [3]:
import shutil, os, re

# Schema required for streaming
schema = T.StructType([T.StructField("request", T.StringType(), True)])

stream_input = (spark.readStream
    .format("csv")
    .option("header", "true")
    .schema(schema)
    .load("data/weblog_src"))

print("Is streaming:", stream_input.isStreaming)
stream_input.printSchema()

Is streaming: True
root
 |-- request: string (nullable = true)



In [4]:
# Write to in-memory table -- trigger(availableNow=True) processes all files then stops
spark.conf.set("spark.sql.streaming.checkpointLocation", "tmp/chk1")
if os.path.exists("tmp/chk1"): shutil.rmtree("tmp/chk1")

query = (stream_input.writeStream
    .format("memory")
    .queryName("weblogs")
    .outputMode("append")
    .trigger(availableNow=True)
    .start())

query.awaitTermination()
print("Query finished")

[Stage 1:=============================>                            (5 + 5) / 10]

Query finished


In [5]:
spark.sql("SELECT COUNT(*) AS total_requests FROM weblogs").show()
spark.sql("SELECT * FROM weblogs LIMIT 5").show(truncate=False)

+--------------+
|total_requests|
+--------------+
|        125581|
+--------------+

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|request                                                                                                                                                                                                                             |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|"6.76.114.84 - - [20/Dec/2019:00:04:22 -0700] ""GET /app/main/posts HTTP/1.0"" 200 4923 ""http://www.huff-mccarthy.com/post/"" ""Mozilla/5.0 (X11; Linux i686) AppleWebKit/5310 (KHTML                                      

In [7]:
# Write to console - continous
spark.conf.set("spark.sql.streaming.checkpointLocation", "tmp/chk1")
if os.path.exists("tmp/chk1"): shutil.rmtree("tmp/chk1")

query = (stream_input.writeStream
    .format("console")
    .queryName("weblogsConsole")
    .outputMode("append")
    .start())

query.awaitTermination(timeout=10)
query.stop()
print("Query finished")

-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------+
|             request|
+--------------------+
|"6.76.114.84 - - ...|
|"252.133.134.245 ...|
|"107.236.220.148 ...|
|"96.203.69.141 - ...|
|"28.6.133.248 - -...|
|"209.78.90.193 - ...|
|"47.128.55.73 - -...|
|"123.126.215.150 ...|
|"154.90.176.136 -...|
|"55.28.147.127 - ...|
|"86.64.8.137 - - ...|
|"125.235.48.31 - ...|
|"62.81.223.157 - ...|
|"44.78.39.95 - - ...|
|"121.97.166.76 - ...|
|"194.12.77.211 - ...|
|"226.233.51.161 -...|
|"206.76.154.42 - ...|
|"159.180.242.53 -...|
|"168.191.195.90 -...|
+--------------------+
only showing top 20 rows
-------------------------------------------
Batch: 1
-------------------------------------------
+--------------------+
|             request|
+--------------------+
|"253.124.169.3 - ...|
|"33.113.171.208 -...|
|"221.232.221.29 -...|
|"145.128.10.87 - ...|
|"116.152.191.124 ...|
|"153.67.190.99 - ...|
|"172.187.167.103 ...|

# Example 2: Extract IP Addresses from Log Lines

We enrich the stream: extract the IP address from each log line using `regexp_extract`.  
This demonstrates applying a **transformation** to a streaming DataFrame.

In [11]:
if os.path.exists("tmp/chk2"): shutil.rmtree("tmp/chk2")
spark.conf.set("spark.sql.streaming.checkpointLocation", "tmp/chk2")

ip_pattern = r"(\d+\.\d+\.\d+\.\d+)"

stream_with_ip = stream_input.withColumn(
    "ip_address", F.regexp_extract(F.col("request"), ip_pattern, 1)
)

query2 = (stream_with_ip.writeStream
    .format("memory")
    .queryName("web_ips")
    .outputMode("append")
    .trigger(availableNow=True)
    .start())

query2.awaitTermination()
spark.sql("SELECT * FROM web_ips LIMIT 5").show(truncate=True)

+--------------------+---------------+
|             request|     ip_address|
+--------------------+---------------+
|"6.76.114.84 - - ...|    6.76.114.84|
|"252.133.134.245 ...|252.133.134.245|
|"107.236.220.148 ...|107.236.220.148|
|"96.203.69.141 - ...|  96.203.69.141|
|"28.6.133.248 - -...|   28.6.133.248|
+--------------------+---------------+



In [12]:
# Top 10 IPs by request count
spark.sql('SELECT ip_address, COUNT(*) AS requests FROM web_ips WHERE ip_address != "" GROUP BY ip_address ORDER BY requests DESC LIMIT 10').show()

+---------------+--------+
|     ip_address|requests|
+---------------+--------+
| 140.148.59.156|      15|
|  85.255.235.53|      15|
|  84.70.133.159|      15|
|  251.213.63.14|      15|
| 72.219.229.184|      15|
| 252.226.194.27|      15|
|103.110.182.239|      15|
|    224.64.1.31|      15|
|  38.160.103.63|      15|
|  211.29.164.12|      15|
+---------------+--------+



In [14]:
query3 = (stream_with_ip.writeStream
    .format("console")
    .queryName("web_ips_console")
    .outputMode("append")
    .start())

query3.awaitTermination(timeout=10)
query3.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+--------------------+---------------+
|             request|     ip_address|
+--------------------+---------------+
|"6.76.114.84 - - ...|    6.76.114.84|
|"252.133.134.245 ...|252.133.134.245|
|"107.236.220.148 ...|107.236.220.148|
|"96.203.69.141 - ...|  96.203.69.141|
|"28.6.133.248 - -...|   28.6.133.248|
|"209.78.90.193 - ...|  209.78.90.193|
|"47.128.55.73 - -...|   47.128.55.73|
|"123.126.215.150 ...|123.126.215.150|
|"154.90.176.136 -...| 154.90.176.136|
|"55.28.147.127 - ...|  55.28.147.127|
|"86.64.8.137 - - ...|    86.64.8.137|
|"125.235.48.31 - ...|  125.235.48.31|
|"62.81.223.157 - ...|  62.81.223.157|
|"44.78.39.95 - - ...|    44.78.39.95|
|"121.97.166.76 - ...|  121.97.166.76|
|"194.12.77.211 - ...|  194.12.77.211|
|"226.233.51.161 -...| 226.233.51.161|
|"206.76.154.42 - ...|  206.76.154.42|
|"159.180.242.53 -...| 159.180.242.53|
|"168.191.195.90 -...| 168.191.195.90|
+-----

# Example 3: Advanced -- Multiple Output Streams

A single streaming source can feed multiple sinks simultaneously.  
Each sink can have a different transformation and output mode.

Here we define two queries on the same input:
1. **Echo**: write raw log lines to a CSV file
2. **IP extract**: extract and store IP addresses separately

In [15]:
if os.path.exists("tmp/chk3"): shutil.rmtree("tmp/chk3")
if os.path.exists("tmp/chk4"): shutil.rmtree("tmp/chk4")
if os.path.exists("tmp/logs_out"): shutil.rmtree("tmp/logs_out")
if os.path.exists("tmp/ips_out"): shutil.rmtree("tmp/ips_out")

spark.conf.set("spark.sql.streaming.checkpointLocation", "tmp/chk3")

# Query 1: echo to CSV
q1 = (stream_input.writeStream
    .format("csv")
    .queryName("echo")
    .option("path", "tmp/logs_out")
    .option("checkpointLocation", "tmp/chk3")
    .outputMode("append")
    .start())

# Query 2: extract IPs to separate CSV
q2 = (stream_input
    .withColumn("ip", F.regexp_extract(F.col("request"), ip_pattern, 1))
    .where(F.col("ip") != "")
    .writeStream
    .format("csv")
    .queryName("ips")
    .option("path", "tmp/ips_out")
    .option("checkpointLocation", "tmp/chk4")
    .outputMode("append")
    .start())

q1.awaitTermination(timeout=15)
q2.awaitTermination(timeout=15)
q1.stop()
q2.stop()
print("Both queries finished")
print("Log files:", len(os.listdir("tmp/logs_out")))
print("IP files: ", len(os.listdir("tmp/ips_out")))

Both queries finished
Log files: 25
IP files:  25


# Example 4: Socket Streaming (Output Modes Demo)

Socket streaming requires an external netcat server -- run in a terminal:
```bash
# macOS/Linux
nc -lk 9999

# Windows
ncat -lk 9999
```

The code below shows the pattern for each output mode.  
**Socket-based queries block indefinitely** -- run them from a `.py` script, not a notebook cell.

In [16]:
input_stream = (spark.readStream.format("socket").option("host", STREAM_HOST).option("port", 9999).load())

In [17]:
q_append = (input_stream
    .select(F.upper(F.col("value")).alias("value"))
    .writeStream.format("console").outputMode("append").start()
    )
q_append.awaitTermination(30)
q_append.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----+
|value|
+-----+
+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+-----+
|value|
+-----+
|  SDF|
+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+-----+
|value|
+-----+
|  SDF|
+-----+

-------------------------------------------
Batch: 3
-------------------------------------------
+------+
| value|
+------+
|SDFSDF|
+------+

-------------------------------------------
Batch: 4
-------------------------------------------
+-----+
|value|
+-----+
|  SDF|
+-----+

-------------------------------------------
Batch: 5
-------------------------------------------
+-----+
|value|
+-----+
|  SDF|
+-----+

-------------------------------------------
Batch: 6
-------------------------------------------
+-----+
|value|
+-----+
|  SDF|
+-----+

-------------------------------------

AttributeError: 'bool' object has no attribute 'stop'

In [22]:
word_counts = (input_stream
    .withColumn("word", F.explode(F.split(F.lower(F.col("value")), r"\W+")))
    .where(F.length("word") > 0)
    .groupBy("word").count())
q_complete = word_counts.writeStream.format("console").outputMode("complete").trigger(processingTime="10 seconds").start()
q_complete.awaitTermination(30)
q_complete.stop()

-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
+----+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
| sdf|    1|
+----+-----+



-------------------------------------------
Batch: 2
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
| sdg|    1|
|sdgf|    1|
|  ef|    1|
| wfe|    1|
|sdgs|    1|
|fsdf|    1|
| dfs|    1|
| sgd|    2|
|  er|    1|
|sdfw|    1|
|  sd|    1|
| sdf|    7|
| wef|    1|
|  fs|    1|
| asd|    1|
|  gw|    1|
+----+-----+



-------------------------------------------
Batch: 3
-------------------------------------------
+-----+-----+
| word|count|
+-----+-----+
|  sdg|    1|
| sdgf|    1|
|dfsdf|    1|
|   ef|    1|
|  wfe|    1|
| sdgs|    1|
|  dfs|    1|
| fsdf|    2|
|  sgd|    2|
|   er|    1|
| sdfw|    1|
|  sgs|    1|
|   sd|    1|
|  sdf|   10|
|  wef|    1|
|   fs|    1|
|  asd|    1|
|   fr|    1|
|   gw|    1|
|   we|    1|
+-----+-----+



# Shutdown Spark when done

In [23]:
spark.stop()